```
- Copyright 2023 DeepMind Technologies Limited
- All software is licensed under the Apache License, Version 2.0 (Apache 2.0); you may not use this file except in compliance with the Apache 2.0 license. You may obtain a copy of the Apache 2.0 license at: https://www.apache.org/licenses/LICENSE-2.0
- All other materials are licensed under the Creative Commons Attribution 4.0 International License (CC-BY).  You may obtain a copy of the CC-BY license at: https://creativecommons.org/licenses/by/4.0/legalcode
- Unless required by applicable law or agreed to in writing, all software and materials distributed here under the Apache 2.0 or CC-BY licenses are distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the licenses for the specific language governing permissions and limitations under those licenses.
- This is not an official Google product
```

# Symmetric admissible set

This notebook contains:
1. the *skeleton* we used for X-evolve to discover large symmetric admissible sets,
2. the *prompt* we used for this problem to instruct large language model,
3. the *functions* discovered by X-evolve that construct those symmetric admissible sets.

## Skeleton

The framework of the *skeleton* we used for X-evolve is from [FunSearch](https://github.com/google-deepmind/funsearch); however, several modules have been reimplemented in C++ to improve the efficiency of evaluation.

In [ ]:
"""Finds large symmetric admissible sets."""

import itertools
import numpy as np


def expand_admissible_set(
    pre_admissible_set: list[tuple[int, ...]],
) -> list[tuple[int, ...]]:
    """Expands a pre-admissible set into an admissible set."""
    num_groups = len(pre_admissible_set[0])
    admissible_set = []
    for row in pre_admissible_set:
        rotations = [[] for _ in range(num_groups)]
        for i in range(num_groups):
            x, y, z = TRIPLES[row[i]]
            rotations[i].append((x, y, z))
            if not x == y == z:
                rotations[i].append((z, x, y))
                rotations[i].append((y, z, x))
        product = list(itertools.product(*rotations))
        concatenated = [sum(xs, ()) for xs in product]
        admissible_set.extend(concatenated)
    return admissible_set


def solve(n: int, w: int) -> tuple[np.ndarray, np.ndarray]:
    """Generates a symmetric constant-weight admissible set I(n, w)."""
    num_groups = n // 3
    assert 3 * num_groups == n
    import cpp_helper
    
    valid_children = np.load(f'admissible_set_{n_w_dim}.npy')
    valid_children_expand = np.load(f'admissible_set_{n_w_dim}_expand.npy')
    valid_children_expand = [tuple(xs) for xs in valid_children_expand.tolist()]
    
    valid_scores = np.array(
        [priority(xs) for xs in valid_children_expand]
    )

    pre_admissible_set = cpp_helper.greedy_search(
        num_groups, valid_scores, valid_children
    )
    return pre_admissible_set, np.array(expand_admissible_set(pre_admissible_set))


# @funsearch.run
def evaluate(kargs) -> int:
    """Returns the size of the expanded admissible set."""
    _, admissible_set = solve(kargs['n'], kargs['w'])
    return len(admissible_set)

# @funsearch.evolve
def priority(el: tuple[int, ...]) -> float:
    """Returns the priority with which we want to add `el` to the set."""
    return 0.0

## Prompt

The *prompt* we used for X-evolve to discover large symmetric admissible sets.

In [ ]:
f'''I'm working on the constant-weight admissible set problem with dimension {n_dim} and weight {w_dim}, using a greedy algorithm that relies on a priority function to determine the vector selection order.


## What I Need
1. **BOLD EVOLUTION OF PRIORITY FUNCTION**: Please create an improved `priority_new` function that might outperform my reference implementations. Don't be constrained by my current approaches - take risks and suggest radically different strategies that might lead to breakthroughs.
2. **MARK ALL TUNABLE PARAMETERS**: For every element in the `priority_new` function that could potentially be tuned, wrap it with tunable([option1, option2, ...]).
  Format examples:
    - `if x == tunable([x1, x2, x3]):`
    - `z = tunable([x + y, x * (y + 1)])`


## Task Description
Please help me develop an improved `priority_new` function by analyzing my reference implementations.
Output Python code only, without any comments.
The score is computed based on the relationships among el[i], el[-i], el[(i - k) % n], and el[(i + k) % n].


## Current Priority Functions
Below are two reference priority functions I've developed.

import itertools
import numpy as np


def expand_admissible_set(
    pre_admissible_set: list[tuple[int, ...]],
    TRIPLES
) -> list[tuple[int, ...]]:
    """Expands a pre-admissible set into an admissible set."""
    num_groups = len(pre_admissible_set[0])
    admissible_set = []
    for row in pre_admissible_set:
        rotations = [[] for _ in range(num_groups)]
        for i in range(num_groups):
            x, y, z = TRIPLES[row[i]]
            rotations[i].append((x, y, z))
            if not x == y == z:
                rotations[i].append((z, x, y))
                rotations[i].append((y, z, x))
        product = list(itertools.product(*rotations))
        concatenated = [sum(xs, ()) for xs in product]
        admissible_set.extend(concatenated)
    return admissible_set

def solve(n: int, w: int) -> tuple[np.ndarray, np.ndarray]:
    """Generates a symmetric constant-weight admissible set I(n, w)."""
    
    TRIPLES = [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 2), (0, 2, 1), (1, 1, 1), (2, 2, 2)]
    INT_TO_WEIGHT = [0, 1, 1, 2, 2, 3, 3]
    
    num_groups = n // 3
    assert 3 * num_groups == n
    import cpp_helper

    # # Compute the scores of all valid (weight w) children.
    # valid_children = []
    # for child in itertools.product(range(7), repeat=num_groups):
    #     weight = sum(INT_TO_WEIGHT[x] for x in child)
    #     if weight == w:
    #         valid_children.append(np.array(child, dtype=np.int32))

    valid_children = np.load('admissible_set_{n_w_dim}.npy')
    valid_children_expand = np.load('admissible_set_{n_w_dim}_expand.npy')
    valid_children_expand = [tuple(xs) for xs in valid_children_expand.tolist()]

    valid_scores = np.array(
        [priority(xs) for xs in valid_children_expand]
    )

    pre_admissible_set = cpp_helper.greedy_search(
        num_groups, valid_scores, valid_children
    )
    return pre_admissible_set, np.array(expand_admissible_set(pre_admissible_set, TRIPLES))


@funsearch.run
def evaluate(kargs) -> int:
    """Returns the size of the expanded admissible set."""
    _, admissible_set = solve(kargs['n'], kargs['w'])
    return len(admissible_set)


@funsearch.evolve
def priority(el: tuple[int, ...]) -> float:
    """Computes a priority score for an element to determine its order of addition to the admissible set.
    
    Args:
        el: A tuple representing a vector with n={n_dim} positions, where each position can be 0, 1, or 2. The element has a weight w={w_dim}, meaning it contains {w_dim} non-zero values.

    Return:
        A float score where higher values indicate higher priority for inclusion in the admissible set.
    """
    n = {n_dim}
    w = {w_dim}
    return 0
'''

## Discovered function that builds a size $43\,650$ admissible set in $A(21, 15)$


In [1]:
import itertools
import numpy as np


def expand_admissible_set(
    pre_admissible_set: list[tuple[int, ...]],
    TRIPLES
) -> list[tuple[int, ...]]:
    """Expands a pre-admissible set into an admissible set."""
    num_groups = len(pre_admissible_set[0])
    admissible_set = []
    for row in pre_admissible_set:
        rotations = [[] for _ in range(num_groups)]
        for i in range(num_groups):
            x, y, z = TRIPLES[row[i]]
            rotations[i].append((x, y, z))
            if not x == y == z:
                rotations[i].append((z, x, y))
                rotations[i].append((y, z, x))
        product = list(itertools.product(*rotations))
        concatenated = [sum(xs, ()) for xs in product]
        admissible_set.extend(concatenated)
    return admissible_set

def solve(n: int, w: int) -> tuple[np.ndarray, np.ndarray]:
    """Generates a symmetric constant-weight admissible set I(n, w)."""
    
    TRIPLES = [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 2), (0, 2, 1), (1, 1, 1), (2, 2, 2)]
    INT_TO_WEIGHT = [0, 1, 1, 2, 2, 3, 3]
    
    num_groups = n // 3
    assert 3 * num_groups == n
    import cpp_helper

    valid_children = np.load('admissible_set_21_15.npy')
    valid_children_expand = np.load('admissible_set_21_15_expand.npy')
    valid_children_expand = [tuple(xs) for xs in valid_children_expand.tolist()]

    valid_scores = np.array(
        [priority(xs, n=n, w=w) for xs in valid_children_expand]
    )

    pre_admissible_set = cpp_helper.greedy_search(
        num_groups, valid_scores, valid_children
    )
    return pre_admissible_set, np.array(expand_admissible_set(pre_admissible_set, TRIPLES))


# @funsearch.run
def evaluate(kargs) -> int:
    """Returns the size of the expanded admissible set."""
    _, admissible_set = solve(kargs['n'], kargs['w'])
    return len(admissible_set)

def priority(el: tuple[int, ...], n: int, w: int) -> float:
    n = 21
    w = 15
    score = 0.0
    for i, val in enumerate(el):
        if val == 1:
            score += 2.5
        elif val == 2:
            score += 5.0
        else:
            score += -0.8
        if i % 3 == 0:
            score += 2.0
        if i % 5 == 0:
            score += -1.2
        if i % 7 == 0:
            score += 1.4
    if sum(el) == w:
        score *= 1.6
    else:
        score *= 1.8
    score += 0.6 * (w - sum(1 for x in el if x != 0))
    score += 0.0 * sum(el[i] * el[(i + 1) % n] for i in range(n))
    score -= 1.6 * sum(el[i] * el[(i + 2) % n] for i in range(n))
    score += 0.9 * sum(el[i] * el[(i + 3) % n] for i in range(n))
    score -= 0.6 * sum(el[i] * el[(i + 4) % n] for i in range(n))
    score += 0.9 * sum(el[i] * el[(i + 5) % n] for i in range(n))
    score -= 0.5 * sum(el[i] * el[(i + 6) % n] for i in range(n))
    score += 0.4 * sum(el[i] * el[(i + 7) % n] for i in range(n))
    score -= 0.8 * sum(el[i] * el[(i + 8) % n] for i in range(n))
    score += 0.8 * sum(el[i] * el[(i + 9) % n] for i in range(n))
    score -= 0.9 * sum(el[i] * el[(i + 10) % n] for i in range(n))
    score += 0.7 * sum(el[i] * el[(i + 11) % n] for i in range(n))
    score -= 0.3 * sum(el[i] * el[(i + 12) % n] for i in range(n))
    score += 0.4 * sum(el[i] * el[(i + 13) % n] for i in range(n))
    score -= 0.7 * sum(el[i] * el[(i + 14) % n] for i in range(n))
    score += 0.9 * sum(el[i] * el[(i + 15) % n] for i in range(n))
    score -= 1.1 * sum(el[i] * el[(i + 16) % n] for i in range(n))
    score += 0.5 * sum(el[i] * el[(i + 17) % n] for i in range(n))
    score -= 0.5 * sum(el[i] * el[(i + 18) % n] for i in range(n))
    score += 0.6 * sum(el[i] * el[(i + 19) % n] for i in range(n))
    score -= 0.9 * sum(el[i] * el[(i + 20) % n] for i in range(n))
    if sum(el) == 14:
        score += 3.5
    else:
        score -= 4.0
    return score

def compute_capacity_bound(n: int, w: int, size: int, m: int) -> float:
  """Returns the lower bound on the cap set capacity.

  We use discovered admissible sets A(n, w) to construct large cap sets,
  following a recipe analogous to [Edel, 2004] and [Tyrrell, 2022]:
  1. Start with the extendable collection E1 = (A0, A1, A2) of three
     6-dimensional cap sets of respective sizes (a0, a1, a2) = (12, 112, 112).
  2. Apply a recursively admissible set I(m, m - 1) to E1, which results in a
     new extendable collection E2 = (B0, B1, B2) of three 6*m-dimensional cap
     sets of sizes (b0, b1, b2) = (a0 * m * a1 ** (m - 1), a1 ** m, a1 ** m).
  3. Apply the admissible set A(n, w) of size `size` to E2, which results in a
     6*m*n-dimensional cap set C of size `size * (b0 ** (n - w)) * (b1 ** w)`.

  Args:
    n: Dimensionality of the discovered admissible set A(n, w).
    w: The weight of the vectors in the discovered admissible set A(n, w).
    size: The size |A(n, w)| of the discovered admissible set.
    m: Dimensionality of the recursively admissible set I(m, m - 1) to use.
  """
  a0, a1, _ = (12, 112, 112)
  b0 = m * a0 * (a1 ** (m - 1))
  b1 = a1 ** m
  log_cap_set_size = np.log(size) + (n - w) * np.log(b0) + w * np.log(b1)
  log_capacity = log_cap_set_size / (6 * m * n)
  return np.exp(log_capacity)

print(evaluate({'n': 21, 'w': 15}))
print(compute_capacity_bound(21, 15, 43650, m=4))

43650
2.2200460296316793


## Discovered function that builds a size $1\,270\,863$ admissible set in $A(27, 19)$

This admissible set implies a new state-of-the-art lower bound of $2.220308$ on the cap set capacity.

In [ ]:
import itertools
import numpy as np


def expand_admissible_set(
    pre_admissible_set: list[tuple[int, ...]],
    TRIPLES
) -> list[tuple[int, ...]]:
    """Expands a pre-admissible set into an admissible set."""
    num_groups = len(pre_admissible_set[0])
    admissible_set = []
    for row in pre_admissible_set:
        rotations = [[] for _ in range(num_groups)]
        for i in range(num_groups):
            x, y, z = TRIPLES[row[i]]
            rotations[i].append((x, y, z))
            if not x == y == z:
                rotations[i].append((z, x, y))
                rotations[i].append((y, z, x))
        product = list(itertools.product(*rotations))
        concatenated = [sum(xs, ()) for xs in product]
        admissible_set.extend(concatenated)
    return admissible_set

def solve(n: int, w: int) -> tuple[np.ndarray, np.ndarray]:
    """Generates a symmetric constant-weight admissible set I(n, w)."""
    
    TRIPLES = [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 2), (0, 2, 1), (1, 1, 1), (2, 2, 2)]
    INT_TO_WEIGHT = [0, 1, 1, 2, 2, 3, 3]
    
    num_groups = n // 3
    assert 3 * num_groups == n
    import cpp_helper

    valid_children = np.load('admissible_set_27_19.npy')
    valid_children_expand = np.load('admissible_set_27_19_expand.npy')
    valid_children_expand = [tuple(xs) for xs in valid_children_expand.tolist()]

    valid_scores = np.array(
        [priority(xs, n=n, w=w) for xs in valid_children_expand]
    )

    pre_admissible_set = cpp_helper.greedy_search(
        num_groups, valid_scores, valid_children
    )
    return pre_admissible_set, np.array(expand_admissible_set(pre_admissible_set, TRIPLES))


# @funsearch.run
def evaluate(kargs) -> int:
    """Returns the size of the expanded admissible set."""
    _, admissible_set = solve(kargs['n'], kargs['w'])
    return len(admissible_set)


def compute_capacity_bound(n: int, w: int, size: int, m: int) -> float:
  """Returns the lower bound on the cap set capacity.

  We use discovered admissible sets A(n, w) to construct large cap sets,
  following a recipe analogous to [Edel, 2004] and [Tyrrell, 2022]:
  1. Start with the extendable collection E1 = (A0, A1, A2) of three
     6-dimensional cap sets of respective sizes (a0, a1, a2) = (12, 112, 112).
  2. Apply a recursively admissible set I(m, m - 1) to E1, which results in a
     new extendable collection E2 = (B0, B1, B2) of three 6*m-dimensional cap
     sets of sizes (b0, b1, b2) = (a0 * m * a1 ** (m - 1), a1 ** m, a1 ** m).
  3. Apply the admissible set A(n, w) of size `size` to E2, which results in a
     6*m*n-dimensional cap set C of size `size * (b0 ** (n - w)) * (b1 ** w)`.

  Args:
    n: Dimensionality of the discovered admissible set A(n, w).
    w: The weight of the vectors in the discovered admissible set A(n, w).
    size: The size |A(n, w)| of the discovered admissible set.
    m: Dimensionality of the recursively admissible set I(m, m - 1) to use.
  """
  a0, a1, _ = (12, 112, 112)
  b0 = m * a0 * (a1 ** (m - 1))
  b1 = a1 ** m
  log_cap_set_size = np.log(size) + (n - w) * np.log(b0) + w * np.log(b1)
  log_capacity = log_cap_set_size / (6 * m * n)
  return np.exp(log_capacity)


In [4]:
def priority(el: tuple[int, ...], n: int, w: int) -> float:
    n = 27
    w = 19
    k_values = [1, 2, 2, 3, 4, 4, 2, 5, 5, 5]
    score = 0.0
    count_1 = sum(0.7 if x == 1 else 0 for x in el)
    count_2 = sum(1.0 if x == 2 else 0 for x in el)
    count_0 = sum(0.5 if x == 0 else 0 for x in el)
    for i in range(n):
        if el[i] == 1:
            score += 0.1 * (el[(n - i) % n] == 2)
            score += 4.3 * (el[(i - k_values[0]) % n] == 0)
            score += 4.5 * (el[(i + k_values[0]) % n] == 2)
            score += 0.5 * (el[(i + k_values[1]) % n] == el[i])
            score += -1.0 * (el[(i - k_values[1]) % n] == el[(i + k_values[1]) % n])
            score += 0.6 * ((el[(i + k_values[0]) % n] + el[(i - k_values[0]) % n]) == 3)
            score += -0.8 * (el[(i + k_values[2]) % n] == el[(i - k_values[2]) % n])
            score += 1.0 * (el[(i + 2 * k_values[0]) % n] == 2)
            score += 1.0 * (el[(i - 2 * k_values[0]) % n] == 0)
            score += 1.0 * (el[(i + k_values[0] + k_values[1]) % n] == el[i])
            score += 0.2 * (el[(i - k_values[0] - k_values[1]) % n] == el[(i + k_values[0] + k_values[1]) % n])
            score += 0.4 * (el[(i + k_values[3]) % n] == 1)
            score += 0.4 * (el[(i - k_values[3]) % n] == 1)
            score += 0.3 * (el[(i + k_values[4]) % n] == 2)
            score += 0.7 * (el[(i - k_values[4]) % n] == 2)
            score += 0.1 * (el[(i + k_values[5]) % n] == 0)
            score += -0.1 * (el[(i - k_values[5]) % n] == 0)
            score += 0.7 * (el[(i + k_values[6]) % n] == 1)
            score += 0.1 * (el[(i + k_values[7]) % n] == 0)
            score += 0.9 * (el[(i + k_values[8]) % n] == 2)
            score += 0.1 * (el[(i + k_values[9]) % n] == 0)
    normalization = w ** 0.6
    score /= normalization
    score += 0.4 * (count_1 / w)
    score += 0.8 * (count_2 / (n - w))
    score += 0.2 * (count_0 / (n - w))
    score *= 2.2
    score += 0.4 * ((count_1 + count_2) / n)
    score += 1.6 * (sum(1 for x in el if x == 0) / n)
    return score

print(evaluate({'n': 27, 'w': 19}))

1270863


In [5]:
print(compute_capacity_bound(27, 19, 1270863, m=4))

2.220308488509796
